# 07 — Machine Learning Pipeline & Cross-Validated Hyperparameter Tuning
**Project:** Axlero Solutions — Credit Risk Scoring and Loan Default Analysis  
**Module:** Model Training (Module 5 — Sowmya)

## 1. Experimental Design
- **Stratified Split:**
  - Training Set: `22,691` loans (`70.0%`, default rate `21.87%`)
  - Validation Set: `4,862` loans (`15.0%`, default rate `21.86%`)
  - Held-Out Test Set: `4,863` loans (`15.0%`, default rate `21.88%`)
- **Models Trained & Compared:**
  1. `Dummy Baseline (Stratified)`
  2. `Logistic Regression (L2)`
  3. `Logistic Regression (Balanced)`
  4. `Decision Tree (Tuned)`
  5. `Random Forest (Tuned)`
  6. `Gradient Boosting (Tuned)`
  7. `XGBoost (Balanced)`
  8. `XGBoost (Champion Tuned)`

In [1]:
import sys
import json
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

summary_data = json.load(open(PROJECT_ROOT / "models" / "all_models_summary.json"))
print("Split Sizes & Stratification Check:")
print(pd.Series(summary_data["split_sizes"]))

print("\n5-Fold Stratified Cross-Validation Hyperparameter Tuning Summary:")
tuning_rows = []
for model_name, info in summary_data["hyperparameter_tuning"].items():
    tuning_rows.append({
        "Model Family": model_name,
        "Best 5-Fold CV ROC-AUC": info["best_cv_roc_auc"],
        "Best Hyperparameters": str(info["best_params"]),
    })
display(pd.DataFrame(tuning_rows))

Split Sizes & Stratification Check:
train_rows                22691.00
val_rows                   4862.00
test_rows                  4863.00
train_default_rate_pct       21.87
val_default_rate_pct         21.86
test_default_rate_pct        21.88
dtype: float64

5-Fold Stratified Cross-Validation Hyperparameter Tuning Summary:


,Model Family,Best 5-Fold CV ROC-AUC,Best Hyperparameters
0,Logistic Regression,0.8990,"{'class_weight': 'balanced', 'C': 2.0}"
1,Decision Tree,0.9086,"{'min_samples_leaf': 40, 'max_depth': 12, 'cri..."
2,Random Forest,0.9255,"{'n_estimators': 250, 'min_samples_leaf': 3, '..."
3,XGBoost,0.9445,"{'subsample': 0.85, 'n_estimators': 350, 'min_..."


## 2. Cross-Validation vs Validation Set Performance Comparison

In [2]:
comp_df = pd.DataFrame(summary_data["model_comparison"])
display(comp_df[["model", "cv_roc_auc_mean", "cv_roc_auc_std", "val_roc_auc", "val_pr_auc", "val_f1"]])

,model,cv_roc_auc_mean,cv_roc_auc_std,val_roc_auc,val_pr_auc,val_f1
0,Gradient Boosting (Tuned),0.9442,0.0061,0.9469,0.9073,0.8415
1,XGBoost (Champion Tuned),0.9438,0.0062,0.9463,0.9045,0.8379
2,XGBoost (Balanced),0.9441,0.0068,0.9451,0.9034,0.8375
3,Random Forest (Tuned),0.9239,0.0077,0.9232,0.8759,0.8155
4,Decision Tree (Tuned),0.9060,0.0105,0.9041,0.8473,0.8117
5,Logistic Regression (Balanced),0.8990,0.0069,0.8997,0.7876,0.6984
6,Logistic Regression (L2),0.8968,0.0073,0.8972,0.7997,0.7198
7,Dummy Baseline (Stratified),0.5046,0.0080,0.4961,0.2173,0.2121
